# thermal-safe-rl: quick debug run on Colab

*Runtime → Change runtime type → CPU.* A GPU does not help here.

This notebook runs the tests and one short training run (about a minute), then estimates how long the full training would take. Run the full training on Kaggle (`notebooks/kaggle_train.ipynb`), where it can keep going after you close the browser.

To keep outputs after the session ends, set `USE_DRIVE = True` (asks for Google Drive access).

In [ ]:
import os
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/thermal-safe-rl/runs"
else:
    OUT = "/content/runs"
if not os.path.exists("/content/thermal-safe-rl"):
    !git clone --depth 1 https://github.com/SugarmanZhu/thermal-safe-rl /content/thermal-safe-rl
%cd /content/thermal-safe-rl
!pip install -q stable-baselines3 gymnasium tabulate
!nproc

In [ ]:
!python -m pytest -q

In [ ]:
import time, subprocess
STEPS = 50_000
t = time.time()
!python train.py --variant ppo_dr --seed 0 --timesteps {STEPS} --out {OUT}
sec = time.time() - t
print(f"{STEPS} steps in {sec:.0f} s -> about {sec * 500_000 / STEPS / 60:.0f} min per 500k-step run on one core")
print("ppo_dr_shieldtrain runs about 3-4x slower (the shield is in the training loop)")

In [ ]:
# Quick check of the trained (very short) policy against two baselines, 2 seeds per scenario.
!python evaluate.py --results {OUT}/results_debug controllers --runs {OUT} --seeds 2 --controllers pid shield_only ppo_dr ppo_dr+shield